In [28]:
import pandas as pd

In [29]:
n1 = pd.read_csv("negativity_analysis_dataset/n1.csv")
n5 = pd.read_csv("negativity_analysis_dataset/n5.csv")

In [30]:
n1.columns

Index(['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat',
       'insult', 'identity_hate'],
      dtype='object')

In [31]:
n5.columns

Index(['Consumer_complaint', 'Product', 'Sentiment', 'Priority'], dtype='object')

In [32]:
n1 = n1.rename(columns={"comment_text": "text"})
n5 = n5.rename(columns={"Consumer_complaint": "text"})

In [33]:
n1.head(2)

,id,text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0


In [34]:
n5.head(2)

,text,Product,Sentiment,Priority
0,I had overdraft protection with Regions Bank i...,Checking or savings account or service,0,1
1,"I am the sole, legal representative of my dece...",Checking or savings account or service,0,1


In [35]:
n1.drop("id" , axis = 1 , inplace = True)

In [36]:
n1["severity"] = (n1["toxic"] + n1["severe_toxic"] + n1["obscene"] + n1["threat"] +n1["insult"] + n1["identity_hate"])

In [37]:
sentiment_map = {
    "Negative": 3,
    "Neutral": 1,
    "Positive": 0
}

priority_map = {
    "High": 3,
    "Medium": 2,
    "Low": 1
}

n5["severity"] = n5["Sentiment"].map(sentiment_map) + n5["Priority"].map(priority_map)

In [38]:
n1 = n1[["text", "severity"]]
n5 = n5[["text", "severity"]]

In [39]:
data = pd.concat([n1, n5], ignore_index=True)

In [41]:
data[data["severity"] == 6]

,text,severity
1017,WOULDN'T BE THE FIRST TIME BITCH. FUCK YOU I'L...,6.0
1312,"SHUT UP, YOU FAT POOP, OR I WILL KICK YOUR ASS!!!",6.0
7299,"You're a stupid cunt \n\nFuck you dumb arse, y...",6.0
13648,Bitch \n\nYou are a little bitch. I fuckin spe...,6.0
13964,I am going to murder ZimZalaBim ST47 for being...,6.0
22158,FUCK YOU!!!!!!!!!!!! YOU FUCKING NIGGER BAG OF...,6.0
29968,u motherfukkin bitch i want to rape you smelly...,6.0
32098,Fuck All Asyriac Nation \n\nQamishli belong to...,6.0
33951,GO FUCK YOURSELF BITCH. I HATE YOUR SOULD. M...,6.0
38513,AM GOING TO RAPE YOU IN THE ASS YOU FAT BITCH ...,6.0


In [44]:
len(data)

161321

In [45]:
data["severity"].value_counts()

severity
0.0    143346
1.0      6360
3.0      4209
2.0      3480
4.0      1760
5.0       385
6.0        31
Name: count, dtype: int64

In [46]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=20000,   stop_words="english", ngram_range=(1,2) )

In [47]:
data["severity"].isna().sum()

np.int64(1750)

In [48]:
data = data.dropna(subset=["severity"])

In [49]:
x = vectorizer.fit_transform(data["text"])
y = data["severity"]

In [50]:
x.shape , y.shape

((159571, 20000), (159571,))

In [51]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=19,stratify=y)

## Ridge Regression

In [53]:
from sklearn.linear_model import Ridge
m1 = Ridge(alpha = 1)
m1.fit(x_train, y_train)

,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",1
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' to shuffle the data.See :term:`Glossary ` for details... versionadded:: 0.17 `random_state` to support Stochastic Average Gradient.",None


In [54]:
y_pred = m1.predict(x_test)

In [55]:
from sklearn.metrics import mean_absolute_error
mae_1 = mean_absolute_error(y_test, y_pred)
print("MAE m1:", mae_1)

MAE m1: 0.20611280767546264


In [57]:
import  numpy as np

In [58]:
y_pred_m1_round = np.clip(np.round(y_pred),0,6)

## LinearSVR

In [59]:
from sklearn.svm import LinearSVR
m2 = LinearSVR()
m2.fit(x_train,y_train)
pred = m2.predict(x_test)

In [61]:
from sklearn.metrics import mean_absolute_error
mae_m2 = mean_absolute_error(y_test, pred)
print("MAE m2:", mae_m2)

MAE m2: 0.12831982848791879


## we will use linearsvr

In [62]:
from sklearn.model_selection import GridSearchCV
param_grid = {
    "C": [0.1, 0.5, 1, 2, 5],
    "epsilon": [0.0, 0.1, 0.2],
    "loss": ["epsilon_insensitive", "squared_epsilon_insensitive"]
}

In [64]:
grid = GridSearchCV(
    m2,
    param_grid,
    cv=3,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

grid.fit(x_train, y_train)

C:\Users\divyadarshee dash\AppData\Roaming\Python\Python313\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LinearSVR()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [0.1, 0.5, ...], 'epsilon': [0.0, 0.1, ...], 'loss': ['epsilon_insensitive', 'squared_epsilon_insensitive']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candid

In [65]:
best_model = grid.best_estimator_

In [67]:
pred = best_model.predict(x_test)
print("Tuned MAE:", mean_absolute_error(y_test, pred))

Tuned MAE: 0.12828075559533075


In [68]:
import joblib
joblib.dump(best_model, "severity_model_linearsvr.joblib")
joblib.dump(vectorizer, "severity_model_vectorizer.joblib")

['severity_model_vectorizer.joblib']

In [21]:
import joblib

In [22]:
v = joblib.load("severity_model_vectorizer.joblib")
m = joblib.load("severity_model_linearsvr.joblib")

In [23]:
def r(Text):
    x_vec = v.transform([Text])
    y_pred = m.predict(x_vec)
    return int(y_pred[0])

In [43]:
print(r(""))

2
